<a href="https://colab.research.google.com/github/Skulfre/Machine-Learning/blob/main/custom_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from pathlib import Path
import requests
import zipfile

data_path = Path('data/')
# Creating a folder & subfolder name
image_path = data_path / "pizza_steak_sushi"

# If image folder doesn't exist, create it, download and upload data
if image_path.is_dir():
  print(f"File path {image_path} already exist")
else:
  print(f"Creating a file path: {image_path} inside of {data_path}...")
  image_path.mkdir(parents=True, exist_ok=True)

  # Download and upload data
  with open(data_path / "pizza_steak_sushi.zip", 'wb') as f: # wb -> Write Binary
     request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
     print(f"Downloading data...")
     f.write(request.content)

  # Unzipping the file:
  with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", 'r') as zip_ref:
    print("Unzipping files...")
    zip_ref.extractall(image_path)

In [ ]:
import os
def walk_through_dir(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    # Dirnames -> Directory Names, File Names -> Images/Files inside Directories,
    # DirPath -> Directory Path
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")
walk_through_dir(image_path)

In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

In [ ]:
import random
from PIL import Image

random.seed(42)
image_path_list = list(image_path.glob("*/*/*.jpg"))
# pizza_steak_sushi/  ← image_path (polazišna tačka)
#     */              ← bilo koji folder (npr. "train" ili "test")
#         */          ← bilo koji folder (npr. "pizza", "steak", "sushi")
#             *.jpg   ← bilo koja .jpg slika

random_image_path = random.choice(image_path_list)
# Choosing a random image path

image_class = random_image_path.parent.stem
# Npr: random_image_path = data/pizza_steak_sushi/train/pizza/123.jpg
# .parent vraća folder u kojem se fajl nalazi (roditelja):
# random_image_path.parent  # → data/pizza_steak_sushi/train/pizza
# .stem vraća ime fajla/foldera bez ekstenzije:
# random_image_path.stem         # → "123"  (samo ime slike)
# random_image_path.parent.stem  # → "pizza" (ime roditeljskog foldera)

img = Image.open(random_image_path)
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

img_to_np = np.asarray(img)
plt.figure(figsize=(10, 7))
plt.imshow(img_to_np)
plt.axis(False)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_transform = transforms.Compose([
    # Resize the image format of 512 * 512 -> 64 * 64
    transforms.Resize(size=(64, 64)),
    # Give it a random flip for data augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    # Transform it into the Tensor
    transforms.ToTensor()
])

In [ ]:
def plot_transformed_image(image_paths, transform, n=3, seed=42):
  # random.seed(seed)
  random_image_paths = random.sample(image_paths, k=n)
  # Since 'image_paths' is a list of random path images we can use .sample
  for image_path in random_image_paths:
    with Image.open(image_path) as f:
      # figure is a whole canvas
      # ax is an axis
      # ax[0] is first half of canvas -> ax[1] second
      fig, ax = plt.subplots(1, 2)
      # Plot original image
      ax[0].imshow(f)
      ax[0].set_title(f"Original size: {f.size}")
      ax[0].axis("off")

      # Transform and plot image
      transformed_image = transform(f).permute(1, 2, 0)
      # Note: permute() will change shape of image to suit matplotlib
      # Basic Image structure is (C, H, W) but Pytorch works with (H, W, C)
      ax[1].imshow(transformed_image)
      ax[1].set_title(f"Transformed size: {transformed_image.shape}")
      ax[1].axis("off")

      fig.suptitle(f"Class: {image_path.parent.stem}")

plot_transformed_image(image_paths=image_path_list,
                       transform=data_transform,
                       n=3,
                       seed=42)

In [ ]:
train_data = datasets.ImageFolder(root=train_dir, # Target folder of Images
                                  transform=data_transform, # How to transoform our Images
                                  target_transform=None) # Transform Labels -> Optional
test_data = datasets.ImageFolder(root=test_dir,
                                 transform=data_transform)
train_data, test_data

In [ ]:
class_names = train_data.classes
# Create a list of classes
class_to_idx = train_data.class_to_idx
# Create a dictionary of classes

class_names, class_to_idx

In [ ]:
len(test_data), len(train_data)

In [ ]:
img, label = train_data[0]
print(f"Image Tensor:\n{img}")
print(f"Image Shape: {img.shape}")
print(f"Image Label: {class_names[label]}")
print(f"Image DataType: {img.dtype}")
print(f"Label DataType: {type(label)}")


In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=1,
                              num_workers=os.cpu_count(),
# num_workers -> How many CPU Cores will be used to operate
# on targeted data -> more = better
# os.cpu_count() using all possible cores and counting them
                              shuffle=True)
test_dataloader = DataLoader(dataset=test_data,
                             batch_size=1,
                             num_workers=os.cpu_count(),
                             shuffle=False)

len(train_dataloader), len(test_dataloader)

In [ ]:
img, label = next(iter(train_dataloader))
img.shape, label

In [ ]:
import os
import pathlib
import torch

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from typing import Tuple, Dict, List

target_direcotry = train_dir
print(f"Path of target_directory: {target_direcotry}")

class_names_list = sorted([directory.name for directory in list(os.scandir(target_direcotry))])
class_names_list

In [ ]:
def find_classes(directory):

  # Sorted vraca values u abecednom redu
  # Recimo da imaš ovakvu strukturu:
  # data/
  # ├── cats/
  # ├── dogs/
  # └── birds/
  class_names_list = sorted(dir.name for dir in list(os.scandir(directory)))
  # → ['birds', 'cats', 'dogs']

  if not class_names_list:
    raise FileNotFoundError("File Not Found, Please check the construction of the data")

  # enumerate daje (index, vrijednost)
  class_names_to_idx = {class_names:i for i, class_names in enumerate(class_names_list)}
  # → {'birds': 0, 'cats': 1, 'dogs': 2}

  return class_names_list, class_names_to_idx
find_classes(train_dir)

In [ ]:
class ImageFolderCustom(Dataset):
  def __init__(self, targ_dir, transform=None):
    super().__init__()

    self.paths = list(Path(targ_dir).glob("*/*.jpg"))
    self.transform = transform
    self.class_names, self.class_to_idx = find_classes(targ_dir)

  def load_image(self, index):
    target_img_path = self.paths[index]
    img = Image.open(target_img_path)
    return img

  def __len__(self):
    return len(self.paths)

  def __getitem__(self, index):
    image = self.load_image(index)
    class_name = self.paths[index].parent.name
    class_to_idx = self.class_to_idx[class_name]

    if self.transform:

      return self.transform(image), class_to_idx
    else:
      # Returnujemo broj klase
      return image, class_to_idx

train_data_custom = ImageFolderCustom(targ_dir=train_dir,
                                      transform=data_transform)

test_data_custom = ImageFolderCustom(targ_dir=test_dir,
                                     transform=data_transform)

train_data_custom, test_data_custom


In [ ]:
len(train_data_custom), len(test_data_custom)

In [ ]:
train_data_custom.class_names

In [ ]:
train_data_custom.class_to_idx

In [ ]:
def display_random_images(dataset: torch.utils.data.Dataset,
                          classes: List[str],
                          n: int,
                          seed: int,
                          display_shape: bool = True):
  if n > 10:
    n = 10
    display_shape = False
    print(f"For display purposes, n shouldn't be larger than 10, setting to 10 and removing shape display.")

  if seed:
    random.seed(seed)

  random_image_samples = random.sample(range(len(dataset)), k=n)

  plt.figure(figsize=(10, 7))

  for i, target_image in enumerate(random_image_samples):

    image, label = dataset[target_image][0], dataset[target_image][1]
    image = image.permute(1, 2, 0)
    plt.subplot(1, n, i+1)
    plt.title(classes[label])
    plt.imshow(image)

    plt.axis('off')
    image, label

display_random_images(dataset=train_data_custom,
                      classes=class_names,
                      n=9,
                      seed=42,
                      display_shape=None)


In [ ]:
image, label = train_data_custom[0]
image

In [ ]:
label

In [ ]:
from torch.utils.data import DataLoader
train_dataloader_custom = DataLoader(dataset=train_data_custom,
                                     batch_size=1,
                                     num_workers=os.cpu_count(),
                                     shuffle=True)
test_dataloader_custom = DataLoader(dataset=test_data_custom,
                                    batch_size=1,
                                    num_workers=os.cpu_count(),
                                    shuffle=False)
image, label = next(iter(train_dataloader_custom))
image.shape, label.shape

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor()
])
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
image_paths = list(Path(train_dir).glob("*/*.jpg"))
plot_transformed_image(image_paths=image_paths,
                       transform=train_transforms,
                       n=10,
                       seed=42)

In [ ]:
simple_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

In [ ]:
train_data_simple = datasets.ImageFolder(root=train_dir, transform=simple_transform, target_transform=None)
test_data_simple = datasets.ImageFolder(root=test_dir, transform=simple_transform)

BATCH_SIZE = 32
NUM_OF_WORKERS = os.cpu_count()

train_dataloader_simple = DataLoader(dataset=train_data_simple,
                                     batch_size=BATCH_SIZE,
                                     num_workers=NUM_OF_WORKERS,
                                     shuffle=True)

test_dataloader_simple = DataLoader(dataset=test_data_simple,
                                    batch_size=BATCH_SIZE,
                                    num_workers=NUM_OF_WORKERS)

train_dataloader_simple, test_dataloader_simple

In [ ]:
from torch import nn

class TinyVGG(nn.Module):
  def __init__(self, input_features, hidden_layers, output_features):
    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input_features, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    self.conv_block_2 = nn.Sequential(
      nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),

      nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
      nn.ReLU(),
      nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layers*16*16, out_features=len(class_names_list))
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # x = self.conv_block_1(x)
    # x = self.conv_block_2(x)
    # x = self.classifier(x)
    # return x
    return self.classifier(self.conv_block_2(self.conv_block_1(x)))
torch.manual_seed(42)

model_0 = TinyVGG(input_features=3, hidden_layers=10, output_features=len(class_names_list)).to(device)

model_0


In [ ]:
# Making predictions without training

image_batch, image_label = next(iter(train_dataloader_simple))
image, label = image_batch[0].unsqueeze(dim=0), image_label[0]
image.shape, label


model_0.to(device)
model_0.eval()
with torch.inference_mode():
  pred = model_0(image.to(device))


print(f"Raw Logits: {pred}")
print(f"Pred Probs: {torch.softmax(pred, dim=1)}")
print(f"Pred Labels: {torch.softmax(pred, dim=1).argmax(dim=1)}")
print(f"Actual Label: {label}")

In [ ]:
!pip install -q torchinfo

In [ ]:
from torchinfo import summary
# Torchinfo -> summary -> Sluzi da vidis strukturu Modela
summary(model=model_0, input_size=[1, 3, 64, 64])

In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):

  model.to(device)

  model.train()

  train_loss = 0

  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    train_logits = model(X)
    train_pred_labels = torch.softmax(train_logits, dim=1).argmax(dim=1)

    loss = loss_fn(train_logits, y)
    train_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  train_loss /= len(dataloader)

  return train_loss

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device):

  model.to(device)

  model.eval()

  test_loss = 0

  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)

      test_logits = model(X)
      test_pred_labels = torch.softmax(test_logits, dim=1).argmax(dim=1)

      loss = loss_fn(test_logits, y)
      test_loss += loss.item()

  test_loss /= len(dataloader)

  return test_loss




In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

In [ ]:
device

In [ ]:
from tqdm.auto import tqdm

def train(model: torch.nn.Module,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          epochs: int,
          device: torch.device):

  results = {
      "train_loss": [],
      "test_loss": [],
  }


  for epoch in tqdm(range(epochs)):

    train_loss = train_step(model = model,
                dataloader = train_dataloader,
                loss_fn = loss_fn,
                optimizer = optimizer,
                device = device)

    test_loss = test_step(model = model,
                dataloader = test_dataloader,
                loss_fn = loss_fn,
                device = device)

    results['train_loss'].append(train_loss)
    results['test_loss'].append(test_loss)

    print(f"Epoch: {epoch+1}/{epochs}, Train Loss: {train_loss}, Test Loss: {test_loss}")

  return results
model_0_results = train(model = model_0, loss_fn = loss_fn, optimizer = optimizer, train_dataloader = train_dataloader_simple, test_dataloader = test_dataloader_simple, epochs = 5, device = device)

In [ ]:
model_0_results.keys() # Vraca kljuceve iz dictionary-a (u ovom slucaju imena train i test loss-a)

In [ ]:
def plot_loss_curves(results: Dict):

  train_loss = results['train_loss']
  test_loss = results['test_loss']
  epochs = range(len(train_loss))
  # print(epochs)

  plt.figure(figsize=(15, 7))

  plt.subplot(1, 2, 1)
  plt.plot(epochs, train_loss, label="Train Loss")
  plt.plot(epochs, test_loss, label="Test Loss")
  plt.title("Loss")
  plt.xlabel("Epoch")
  plt.legend()
plot_loss_curves(model_0_results)

In [ ]:
# Adding Data Augmentation

train_transformation_trivia_augment = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])


train_data_trivia_augment = datasets.ImageFolder(root=train_dir, transform=train_transformation_trivia_augment)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_data_trivia_augment, test_data

In [ ]:
train_dataloader_augmented = DataLoader(
    dataset=train_data_trivia_augment,
    batch_size=32,
    shuffle=True,
    num_workers=os.cpu_count(),
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=32,
    num_workers=os.cpu_count(),
)

train_dataloader_augmented, test_dataloader
len(train_dataloader_augmented), len(test_dataloader)

In [ ]:
torch.manual_seed(42)
model_1 = TinyVGG(input_features=3, hidden_layers=10, output_features=len(class_names_list)).to(device)
model_1

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.1)

In [ ]:
model_1_results = train(model= model_1,
                        loss_fn= loss_fn,
                        optimizer= optimizer,
                        train_dataloader= train_dataloader_augmented,
                        test_dataloader= test_dataloader,
                        epochs= 10,
                        device= device)

In [ ]:
plot_loss_curves(model_1_results)

In [ ]:
import pandas as pd

model_0_pd = pd.DataFrame(model_0_results)
model_1_pd = pd.DataFrame(model_1_results)
model_1_pd


In [ ]:
from pathlib import Path
import requests

custom_image_path = data_path / "custom_image.jpeg"


if not custom_image_path.is_file():
  with open(custom_image_path, 'wb') as f:
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
    print(f"Download a file...")
    f.write(request.content)
else:
  print(f"File already exists...")

In [ ]:
custom_img = custom_image_path
Image.open(custom_img)

In [ ]:
import torchvision

# This method will read a JPEG or PNG image and turn it into a 3 dimensional RGB or grayscale torch.Tensor with values of datatype uint8 in range [0, 255].
custom_image_uint8 = torchvision.io.read_image(str(custom_image_path))

# Print out image data
print(f"Custom image tensor:\n{custom_image_uint8}\n")
print(f"Custom image shape: {custom_image_uint8.shape}\n")
print(f"Custom image dtype: {custom_image_uint8.dtype}")

In [ ]:
custom_img = Image.open(custom_image_path)
custom_data_img = data_transform(custom_img).to(device)
custom_data_img.shape

In [ ]:
model_1.eval()
with torch.inference_mode():
  pred = model_1(custom_data_img.unsqueeze(dim=0))
  pred_label = torch.softmax(pred, dim=1).argmax(dim=1)
print(class_names_list[pred_label])

In [ ]:
def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List,
                        device: torch.device,
                        transform=None,):

  model.to(device)

  image = Image.open(image_path)

  if transform:
    print(f"Transforming an image...")
    image = transform(image).to(device)

  model.eval()
  with torch.inference_mode():
    y_logits = model(image.unsqueeze(dim=0))
    y_pred_labels = torch.softmax(y_logits, dim=1).argmax(dim=1)

  print(f"Prediction Logits: {y_logits}")
  print(f"Prediction Class: {class_names[y_pred_labels]}")


  plt.figure(figsize=(10, 7))
  plt.title(f"Class: {class_names[y_pred_labels]}")
  # Permute because Pytorch works with (C, H, W)
  # MatPlotLib works with (H, W, C)
  plt.imshow(image.permute(1, 2, 0).cpu())
  plt.axis("off")

pred_and_plot_image(model=model_1,
                    image_path=custom_image_path,
                    class_names=class_names_list,
                    transform=data_transform,
                    device = device
                    )